In [ ]:
# Imports
import sys
import logging
from datetime import datetime
import pandas as pd
from IPython.display import display

sys.path.insert(0, '../../../LOGOS')
from src import Pert, plot_gantt_chart, plot_resource_utilization, plot_location_utilization, plot_equipment_utilization
# Configure logging in the runner (avoid setting basicConfig inside the module)
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
import json
from pathlib import Path

cwd = Path.cwd()
benchmark = cwd/'benchmarks'
benchmark_results_file = benchmark/'priority_rules_results.json'

## Load Benchmark results
with open(benchmark_results_file, "r", encoding="utf-8") as f:
    benchmark_data = json.load(f)

In [ ]:
def run_case(case_name, file_name, json_path,
             schema_file="outage_schema.json",
             benchmark_data=None):
    """
    Runs scheduling comparison for a given case and file.

    Parameters:
        case_name (str): Case identifier (e.g., 'j60')
        file_name (str): File name (e.g., 'j601_1.sm')
        json_path (str): Path to JSON file for Pert model
        schema_file (str): Path to schema file (default: outage_schema.json)
        benchmark_data (dict): Benchmark dataset for RCPSP comparison

    Returns:
        results_df (pd.DataFrame): LOGOS.CPM results
        data_df (pd.DataFrame): RCPSP benchmark results
    """

    results_sgs = {}
    results_pgs = {}
    results_pgs_pr = {}


    # Load Pert Model
    pert = Pert.from_json_file(json_path, schema_path=schema_file)

    prs = [
        'es','ef','ls','lf', 'duration','random',
        'mts', 'mtp', 'grpw', 'grd', 'rr', 'avgrr',
        'maxrr', 'minrr', 'irsm','wcs','acs',
        'mehh_8000_b','mehh_3375_b',
        'mehh_1000_b','mehh_125_b','gphh_b'
    ]

    # Compute results with Serial
    for rule in prs:
        out = pert.calculateSerialScheduleWithResources(priority_rule=rule)
        results_sgs[rule] = out['scheduled_duration'] - 2  # remove start/end duration
        violations, is_feasible = pert.check_dependency_violations()
        if not is_feasible:
            print(violations)
    sgs = ['first', 'max_use_res_ranked', 'max_use_res_shuffled', 'md_knapsack', 'look_ahead']

    # Compute results with Parallel
    for s in sgs:
        out = pert.calculateScheduleWithResources(sgs=s)
        results_pgs[s] = out['scheduled_duration'] - 2  # remove start/end duration
        violations, is_feasible = pert.check_dependency_violations()
        if not is_feasible:
            print(violations)

    for s in sgs:
        results_pgs_pr[s] = {}
        for rule in prs:
            out = pert.calculateScheduleWithResources(sgs=s, priority_rule=rule)
            results_pgs_pr[s][rule] = out['scheduled_duration'] - 2  # remove start/end duration
            violations, is_feasible = pert.check_dependency_violations()
            if not is_feasible:
                print(violations)



    print('Results from LOGOS.CPM Using Serial Generation Scheme:')
    print('-' * 60)
    results_df_sgs = pd.DataFrame(results_sgs, index=[0])
    display(results_df_sgs)

    if benchmark_data is None:
        raise ValueError("benchmark_data must be provided")

    data = benchmark_data[case_name][file_name]
    data_df = pd.DataFrame(data, index=[0]).filter(like='serial_forward')
    data_df.columns = data_df.columns.str.replace("_serial_forward", "", regex=False)
    data_df.columns = data_df.columns.str.lower()

    display(data_df)

    print('Results from LOGOS.CPM Using Parallel Generation Scheme:')
    print('-' * 60)
    results_df_pgs = pd.DataFrame(results_pgs, index=[0])
    display(results_df_pgs)

    for s in sgs:
        print(f'Results from LOGOS.CPM Using Parallel Generation Scheme "{s}" with Priority Rule:')
        print('-' * 60)
        results_df_pgs_pr = pd.DataFrame(results_pgs_pr[s], index=[0])
        display(results_df_pgs_pr)

    # RCPSP benchmark comparison
    print('Results from RCPSP')
    print('-' * 60)

    data_df = pd.DataFrame(data, index=[0]).filter(like='parallel_forward')
    data_df.columns = data_df.columns.str.replace("_parallel_forward", "", regex=False)
    data_df.columns = data_df.columns.str.lower()

    display(data_df)


    return results_df_sgs, results_df_pgs, data_df

## Scheduling with 30 activities

In [ ]:
results_df_sgs, results_df_pgs, data_df = run_case(
    case_name='j30',
    file_name='j301_1.sm',
    json_path='j301_1.json',
    benchmark_data=benchmark_data
)


## Scheduling with 60 activities

In [ ]:
results_df_sgs, results_df_pgs, data_df = run_case(
    case_name='j60',
    file_name='j601_1.sm',
    json_path='j601_1.json',
    benchmark_data=benchmark_data
)

## Scheduling with 90 activities

In [ ]:
results_df_sgs, results_df_pgs, data_df = run_case(
    case_name='j90',
    file_name='j901_1.sm',
    json_path='j901_1.json',
    benchmark_data=benchmark_data
)

## Scheduling with 120 activities

In [ ]:
results_df_sgs, results_df_pgs, data_df = run_case(
    case_name='j120',
    file_name='j1201_1.sm',
    json_path='j1201_1.json',
    benchmark_data=benchmark_data
)

## Backward Scheduling (ALAP) and Forward–Backward–Forward (FBF) Improvement

Tests the two new backward SGS methods against the same PSPLIB instances:

| Step | Method | Description |
|------|--------|-------------|
| **Backward Serial** | `calculateBackwardSerialScheduleWithResources` | ALAP serial SGS with each priority rule |
| **Backward Parallel** | `calculateBackwardScheduleWithResources` | ALAP parallel SGS (TF-based priority) |
| **FBF** | F1 → B → F2 | Forward → Backward → Forward improvement (Valls et al. 2005) |

**FBF logic for each priority rule:**
1. **F1** — `calculateSerialScheduleWithResources(rule)` → makespan M₁, activity order O₁
2. **B** — `calculateBackwardSerialScheduleWithResources(M₁, _ordered=O₁)` → ALAP start times
3. **F2** — `calculateSerialScheduleWithResources(_ordered=ALAP order)` → makespan M₂
4. **Δ = M₁ − M₂** (positive = improvement)

In [ ]:
def run_fbf_case(json_path, schema_file="outage_schema.json"):
    """
    Tests Backward Serial SGS, Backward Parallel SGS, and the
    Forward-Backward-Forward (FBF) improvement loop on one PSPLIB benchmark.

    Section 1 — Backward SGS standalone:
        Uses the best forward serial makespan as the ALAP horizon and runs:
        - Backward Serial SGS with each of the 22 priority rules
        - Backward Parallel SGS with default TF-based priority

    Section 2 — FBF improvement:
        For every priority rule, runs the three-step improvement loop and
        reports the per-rule makespans (F1, F2) and change (delta).
    """
    prs = [
        'es', 'ef', 'ls', 'lf', 'duration', 'random',
        'mts', 'mtp', 'grpw', 'grd', 'rr', 'avgrr',
        'maxrr', 'minrr', 'irsm', 'wcs', 'acs',
        'mehh_8000_b', 'mehh_3375_b', 'mehh_1000_b', 'mehh_125_b', 'gphh_b',
    ]

    # ── Section 1a: Forward serial baseline (all rules) ──────────────────────
    fwd_results = {}
    for rule in prs:
        p = Pert.from_json_file(json_path, schema_path=schema_file)
        out = p.calculateSerialScheduleWithResources(priority_rule=rule)
        fwd_results[rule] = out['scheduled_duration'] - 2   # strip dummy START/END

    best_rule = min(fwd_results, key=fwd_results.get)
    best_makespan_h = fwd_results[best_rule] + 2            # include dummies for horizon
    print(f"Best forward serial: {fwd_results[best_rule]:.1f} h  (rule='{best_rule}')")

    # ── Section 1b: Backward Serial SGS (horizon = best forward makespan) ────
    bwd_s_dur  = {}
    bwd_s_comp = {}
    for rule in prs:
        p = Pert.from_json_file(json_path, schema_path=schema_file)
        # out = p.calculateBackwardSerialScheduleWithResources(
        #     makespan_hours=best_makespan_h, priority_rule=rule
        # )
        out = p.calculateBackwardSerialScheduleWithResources(
            makespan_hours=None, priority_rule=rule
        )
        bwd_s_dur[rule]  = out['scheduled_duration'] - 2
        bwd_s_comp[rule] = f"{out['n_completed']}/{out['n_activities']}"

    # ── Section 1c: Backward Parallel SGS (TF-based priority) ────────────────
    p_bp = Pert.from_json_file(json_path, schema_path=schema_file)
    # out_bp = p_bp.calculateBackwardScheduleWithResources(makespan_hours=best_makespan_h)
    out_bp = p_bp.calculateBackwardScheduleWithResources(makespan_hours=None)

    bp_placed = f"{out_bp['n_completed']}/{out_bp['n_activities']}"
    bp_dur    = out_bp['scheduled_duration'] - 2

    # ── Section 2: FBF improvement loop ──────────────────────────────────────
    fbf_rows = {}
    for rule in prs:
        p = Pert.from_json_file(json_path, schema_path=schema_file)

        # F1: derive priority-ordered list, then run forward serial
        raw = p.priority_calculation(list(p.forwardDict.keys()), rule)
        if raw and isinstance(raw[0], tuple):
            f1_ordered = [a for (a, _, _) in raw]
        else:
            f1_ordered = list(raw)

        out_f1 = p.calculateSerialScheduleWithResources(_ordered=f1_ordered)
        m1 = out_f1['scheduled_duration']

        # B: backward serial on the same Pert instance (same Activity objects,
        #    so forwardDict lookups are consistent); resets state internally.
        out_b = p.calculateBackwardSerialScheduleWithResources(
            makespan_hours=m1, _ordered=f1_ordered
        )

        # ALAP ordering: sort placed activities by actual start time (ascending).
        placed = sorted(
            [(a, a.returnAbsTimes()[0]) for a in p.forwardDict
             if a.returnAbsTimes()[0] is not None],
            key=lambda x: x[1]
        )
        alap_ordered = [a for a, _ in placed]
        # Append unplaced activities at the end; F2's precedence logic handles them.
        placed_set = set(alap_ordered)
        for a in f1_ordered:
            if a not in placed_set:
                alap_ordered.append(a)

        # F2: forward serial from the ALAP ordering
        out_f2 = p.calculateSerialScheduleWithResources(_ordered=alap_ordered)
        m2 = out_f2['scheduled_duration']

        fbf_rows[rule] = {
            'F1 (h)': round(m1 - 2, 1),
            'B placed': out_b['n_completed'],
            'F2 (h)': round(m2 - 2, 1),
            'delta (h)': round(m1 - m2, 1),
        }

    # ── Display ──────────────────────────────────────────────────────────────
    print('\nForward Serial SGS — baseline durations (h):')
    print('-' * 60)
    display(pd.DataFrame({'forward_serial': fwd_results}, index=[0]))
    print(fwd_results)

    print('\nBackward Serial SGS — ALAP durations (h) and completion:')
    print('-' * 60)
    df_bwd_s = pd.DataFrame({
        'backward_serial (h)': bwd_s_dur,
        'placed':              bwd_s_comp,
    }).T
    display(df_bwd_s)

    case_name=json_path.split('.')[0][:-3]
    file_name=json_path.split('.')[0] + '.sm'
    data = benchmark_data[case_name][file_name]
    data_df = pd.DataFrame(data, index=[0]).filter(like='serial_backward')
    data_df.columns = data_df.columns.str.replace("_serial_backward", "", regex=False)
    data_df.columns = data_df.columns.str.lower()

    display(data_df)

    print(f'\nBackward Parallel SGS (TF-based):  placed={bp_placed}'
          f'  scheduled_duration={bp_dur:.1f} h')

    print('\nForward-Backward-Forward (FBF) improvement:')
    print('-' * 60)
    df_fbf = pd.DataFrame(fbf_rows).T
    df_fbf['delta (h)'] = df_fbf['delta (h)'].astype(float)
    df_fbf = df_fbf.sort_values('F1 (h)')
    display(df_fbf)

    n_improved = int((df_fbf['delta (h)'] > 0).sum())
    best_delta = float(df_fbf['delta (h)'].max())
    avg_delta  = float(df_fbf['delta (h)'].mean())
    print(
        f"\nFBF improved {n_improved}/{len(prs)} priority rules"
        f" | max improvement = {best_delta:.1f} h"
        f" | avg improvement = {avg_delta:.2f} h"
    )
    return df_fbf

### j30 — Backward SGS and FBF

In [ ]:
df_fbf_j30 = run_fbf_case('j301_1.json')

### j60 — Backward SGS and FBF

In [ ]:
df_fbf_j60 = run_fbf_case('j601_1.json')

### j90 — Backward SGS and FBF

In [ ]:
df_fbf_j90 = run_fbf_case('j901_1.json')

### j120 — Backward SGS and FBF

In [ ]:
df_fbf_j120 = run_fbf_case('j1201_1.json')